# Aplicación RAG con NVIDIA NIM en Google Colab

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.


## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.


In [1]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 4.1 MB/s eta 0:00:00


## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. También puedes introducirla de forma oculta cuando la celda la solicite.


In [10]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

Si quieres usar documentos propios, súbelos a `data/company_documents.json` con los campos `id`, `title`, `content` y, opcionalmente, `metadata`. Si el archivo no existe, la aplicación usa un conjunto de demostración.


In [11]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


company_documents = load_documents(Path("company_documents.json"))
print(f"✅ {len(company_documents)} documentos disponibles")


ℹ️ No se encontró company_documents.json; se usarán documentos de demostración.
✅ 4 documentos disponibles


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión de Colab esté activa.


In [12]:
def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres un asistente empresarial. Responde únicamente con información "
            "del contexto proporcionado. Si el contexto no contiene la respuesta, "
            "indica que no tienes esa información. Responde en español y de forma concisa.",
        ),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks


print("Construyendo el índice...")
rag_pipeline, indexed_chunks = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


Construyendo el índice...


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 4 documentos, 4 chunks


## 5. Crear la interfaz Gradio

In [13]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()
        documents = result["documents"]

        unique_sources = []
        for document in documents:
            title = document.metadata.get("title", "Sin título")
            if title not in unique_sources:
                unique_sources.append(title)

        sources_md = "**Fuentes recuperadas:**\n" + "\n".join(
            f"- {title}" for title in unique_sources
        ) if unique_sources else "No se encontraron documentos relevantes."

    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
    gr.Markdown(
        "# RAG empresarial\n"
        "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown("---")
            gr.Markdown(
                f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                f"**LLM:** `{GENERATION_MODEL}`  \n"
                "**Vector store:** Qdrant en memoria  \n"
                "**Retrieval:** k=3, threshold=0.3"
            )

        with gr.Column(scale=2):
            # Gradio 6 usa mensajes {role, content} por defecto;
            # el antiguo argumento type="messages" ya no existe.
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(
                    placeholder="Haz una pregunta sobre los documentos...",
                    label="Pregunta",
                    lines=2,
                    scale=4,
                )
                send_button = gr.Button("Enviar", variant="primary", scale=1)

            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuál es el precio del plan Enterprise de CloudSync Pro?"],
            ["¿Cuántos días pueden trabajar remotamente los empleados?"],
            ["¿Qué pasa con los reembolsos mayores a $100?"],
            ["¿Qué ocurre durante la primera semana de onboarding?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )

    send_button.click(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    question_box.submit(
        chat,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot, sources_box],
    )
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.


In [14]:
demo.queue().launch(
    share=True,
    debug=True,
    show_error=True,
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b6e82b4a48057a80d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6b6e82b4a48057a80d.gradio.live


In [7]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

modelos = ChatNVIDIA.get_available_models()
for m in modelos:
    print(m.id)

google/gemma-3-27b-it
mistralai/codestral-22b-instruct-v0.1
minimaxai/minimax-m2
qwen/qwq-32b
zyphra/zamba2-7b-instruct
nvidia/llama-3.3-nemotron-super-49b-v1
tiiuae/falcon3-7b-instruct
google/gemma-3-12b-it
z-ai/glm4.7
stepfun-ai/step-3.7-flash
nvidia/nemotron-3-super-120b-a12b
google/gemma-2-2b-it
nvidia/nvclip
nvidia/nemotron-3.5-lightning-30b-a3b
gotocompany/gemma-2-9b-cpt-sahabatai-instruct
nvidia/nemotron-nano-12b-v2-vl
mistralai/mixtral-8x7b-instruct-v0.1
nvidia/riva-translate-4b-instruct
mistralai/mistral-small-4-119b-2603
z-ai/glm-5.1
deepseek-ai/deepseek-v3.1
nvidia/usdcode-llama-3.1-70b-instruct
qwen/qwen3-235b-a22b
meta/llama-guard-4-12b
google/shieldgemma-9b
upstage/solar-10.7b-instruct
deepseek-ai/deepseek-r1-distill-qwen-32b
deepseek-ai/deepseek-v4-flash
nvidia/nemoretriever-parse
microsoft/phi-3-mini-128k-instruct
google/gemma-3-4b-it
google/paligemma
thudm/chatglm3-6b
nvidia/nemotron-4-mini-hindi-4b-instruct
qwen/qwen3-next-80b-a3b-instruct
stepfun-ai/step-3.5-flash
de

In [9]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

candidatos = [
    "google/gemma-4-31b-it",
    "mistralai/mistral-small-4-119b-2603",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "nvidia/nemotron-3-nano-30b-a3b",
    "nvidia/nemotron-3.5-lightning-30b-a3b",
    "qwen/qwen3.5-122b-a10b",
    "deepseek-ai/deepseek-v4-flash",
    "mistralai/mistral-medium-3-instruct",
    "meta/llama-4-maverick-17b-128e-instruct",
]

funcionan = []
for nombre in candidatos:
    try:
        r = ChatNVIDIA(model=nombre, max_completion_tokens=200).invoke("Responde solo con la palabra: listo")
        print("✅", nombre, "→", repr(r.content)[:60])
        funcionan.append(nombre)
    except Exception as e:
        print("❌", nombre, "→", str(e)[:60].replace("\n", " "))

print("\nFuncionan:", funcionan)

❌ google/gemma-4-31b-it → HTTPSConnectionPool(host='integrate.api.nvidia.com', port=44
❌ mistralai/mistral-small-4-119b-2603 → [410] Gone The model 'mistralai/mistral-small-4-119b-2603' h
❌ openai/gpt-oss-120b → [410] Gone The model 'openai/gpt-oss-120b' has reached its e
✅ openai/gpt-oss-20b → 'listo'
❌ nvidia/nemotron-3-nano-30b-a3b → [410] Gone The model 'nvidia/nemotron-3-nano-30b-a3b' has re
✅ nvidia/nemotron-3.5-lightning-30b-a3b → 'listo'
❌ qwen/qwen3.5-122b-a10b → [410] Gone The model 'qwen/qwen3.5-122b-a10b' has reached it
❌ deepseek-ai/deepseek-v4-flash → [410] Gone The model 'deepseek-ai/deepseek-v4-flash' has rea
❌ mistralai/mistral-medium-3-instruct → Extra data: line 1 column 5 (char 4)


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:243: UserWarning: Model mistralai/mistral-medium-3-instruct on endpoint https://integrate.api.nvidia.com/v1 is deprecated and may be removed in a future release.
  warnings.warn(


❌ meta/llama-4-maverick-17b-128e-instruct → [410] Gone The model 'meta/llama-4-maverick-17b-128e-instruc

Funcionan: ['openai/gpt-oss-20b', 'nvidia/nemotron-3.5-lightning-30b-a3b']


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.


In [ ]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()
